# Rusted Spire — RL Experiments

Ironclad (Strike × 5, Defend × 4, Bash × 1) vs various Act 1 enemies.  
Algorithm: MaskablePPO (sb3-contrib).  
Environment: Rust headless simulator exposed via PyO3.

Available enemies: `JawWorm`, `Cultist`, `LouseNormal`, `LouseDefensive`, `GremlinNob`  
Default: `Cultist` at Ascension 7 (~62% random win rate — good training difficulty).

Two environment wrappers:
- **`SpireEnv`** — single combat; pass `enemy=` and `ascension=` to constructor.
- **`MultiCombatEnv`** — chained combats; pass `enemies=[...]` list (one name per slot).

## 1. Imports & sanity check

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

import rusted_spire

print(f"Obs size:    {rusted_spire.OBS_SIZE}")
print(f"Action size: {rusted_spire.ACTION_SIZE}")

Obs size:    77
Action size: 11


## 2. Gymnasium environment

In [ ]:
class SpireEnv(gym.Env):
    """Gymnasium wrapper around the Rust SlayEnv — single combat."""

    metadata = {"render_modes": []}

    def __init__(self, enemy: str = "Cultist", ascension: int = 7):
        super().__init__()
        self._env = rusted_spire.SlayEnv(enemy=enemy, ascension=ascension)
        self._action_mask = np.ones(rusted_spire.ACTION_SIZE, dtype=bool)

        self.observation_space = spaces.Box(
            low=-1.0, high=2.0,
            shape=(rusted_spire.OBS_SIZE,),
            dtype=np.float32,
        )
        self.action_space = spaces.Discrete(rusted_spire.ACTION_SIZE)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        s = seed if seed is not None else random.randint(0, 2**32 - 1)
        obs, mask = self._env.reset(s, None)
        self._action_mask = np.array(mask, dtype=bool)
        return np.array(obs, dtype=np.float32), {}

    def step(self, action: int):
        obs, mask, reward, done = self._env.step(int(action))
        self._action_mask = np.array(mask, dtype=bool)
        return np.array(obs, dtype=np.float32), reward, done, False, {}

    def action_masks(self) -> np.ndarray:
        return self._action_mask

## 2b. Multi-combat environment

Chain fights in one episode. HP carries over; block and powers reset between fights.  
Pass `enemies=["Cultist", "GremlinNob"]` to set a fixed sequence, or a single-element list to repeat.  
Reward accumulates: each victory gives `+1 + hp_ratio`, defeat gives `-1`.  
Max episode reward = `len(enemies) × 2.0` (win everything at full HP).

In [ ]:
class MultiCombatEnv(gym.Env):
    """
    Chain fights in sequence. HP carries over; block and powers reset between fights.
    Reward: +1+hp_ratio per victory, -1 on defeat.
    """

    metadata = {"render_modes": []}

    def __init__(self, enemies: list[str] = None, ascension: int = 7):
        super().__init__()
        self._enemies = enemies or ["Cultist"] * 3
        self._ascension = ascension
        # One SlayEnv per slot so enemy type can vary
        self._envs = [
            rusted_spire.SlayEnv(enemy=e, ascension=ascension)
            for e in self._enemies
        ]
        self._action_mask = np.ones(rusted_spire.ACTION_SIZE, dtype=bool)
        self._combat_idx = 0
        self._base_seed = 0

        self.observation_space = spaces.Box(
            low=-1.0, high=2.0,
            shape=(rusted_spire.OBS_SIZE,),
            dtype=np.float32,
        )
        self.action_space = spaces.Discrete(rusted_spire.ACTION_SIZE)

    def _combat_seed(self) -> int:
        return (self._base_seed + self._combat_idx * 100_003) % (2**32)

    def _current_env(self):
        return self._envs[self._combat_idx]

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._base_seed = seed if seed is not None else random.randint(0, 2**32 - 1)
        self._combat_idx = 0
        obs, mask = self._current_env().reset(self._combat_seed(), None)
        self._action_mask = np.array(mask, dtype=bool)
        return np.array(obs, dtype=np.float32), {}

    def step(self, action: int):
        obs, mask, reward, done = self._current_env().step(int(action))
        self._action_mask = np.array(mask, dtype=bool)

        if not done:
            return np.array(obs, dtype=np.float32), reward, False, False, {}

        if reward < 0:
            return np.array(obs, dtype=np.float32), reward, True, False, {}

        # Victory — carry surviving HP into next combat
        surviving_hp = max(1, round(obs[0] * rusted_spire.MAX_HP))
        self._combat_idx += 1

        if self._combat_idx >= len(self._enemies):
            return np.array(obs, dtype=np.float32), reward, True, False, {}

        next_obs, next_mask = self._current_env().reset(self._combat_seed(), surviving_hp)
        self._action_mask = np.array(next_mask, dtype=bool)
        return np.array(next_obs, dtype=np.float32), reward, False, False, {}

    def action_masks(self) -> np.ndarray:
        return self._action_mask

## 3. Environment smoke test

Run one episode with random legal actions and print the trajectory.

In [3]:
env = SpireEnv()
obs, _ = env.reset(seed=42)

total_reward = 0
steps = 0
done = False

while not done:
    mask = env.action_masks()
    legal = np.where(mask)[0]
    action = int(np.random.choice(legal))
    obs, reward, done, _, _ = env.step(action)
    total_reward += reward
    steps += 1

print(f"Episode ended in {steps} steps  |  total reward: {total_reward:.2f}")

Episode ended in 14 steps  |  total reward: 1.44


## 4. Random policy baseline

Win rate of a policy that always picks a random legal action.

In [ ]:
N = 1000

def random_winrate(env_cls, n=N, **kwargs):
    env = env_cls(**kwargs)
    wins, total_rewards = 0, []
    for i in range(n):
        obs, _ = env.reset(seed=i)
        done, ep_reward = False, 0.0
        while not done:
            legal = np.where(env.action_masks())[0]
            obs, reward, done, _, _ = env.step(int(np.random.choice(legal)))
            ep_reward += reward
        if ep_reward > 0:
            wins += 1
        total_rewards.append(ep_reward)
    return wins / n, np.mean(total_rewards)

wr1, mr1 = random_winrate(SpireEnv)
print(f"Single combat  — win rate: {wr1:.1%}  mean reward: {mr1:.3f}")

wr3, mr3 = random_winrate(MultiCombatEnv, n_combats=3)
print(f"3-combat chain — win rate: {wr3:.1%}  mean reward: {mr3:.3f}  (max possible: 6.0)")

## 5. Train — MaskablePPO

Tweak hyperparameters here before running.

In [ ]:
# --- Hyperparameters ---
TIMESTEPS   = 500_000
N_ENVS      = 8
LEARNING_RATE = 3e-4
N_STEPS     = 512
BATCH_SIZE  = 256
N_EPOCHS    = 10
GAMMA       = 0.99

vec_env  = make_vec_env(SpireEnv, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
eval_env = make_vec_env(SpireEnv, n_envs=4,      vec_env_cls=SubprocVecEnv)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    verbose=1,
    tensorboard_log="runs/",
)

model.learn(total_timesteps=TIMESTEPS)
model.save("models/ppo_spire")
print("Saved to models/ppo_spire")

## 6. Evaluate trained model

In [ ]:
# Load from disk if needed:
# model = MaskablePPO.load("models/ppo_spire", env=SpireEnv())

mean_reward, std_reward = evaluate_policy(
    model,
    SpireEnv(),
    n_eval_episodes=500,
    warn=False,
)
print(f"Mean reward: {mean_reward:.3f} ± {std_reward:.3f}")

# Win rate: reward > 0 means victory
wins = 0
N_EVAL = 500
eval_single = SpireEnv()
for i in range(N_EVAL):
    obs, _ = eval_single.reset(seed=10_000 + i)
    done = False
    while not done:
        action, _ = model.predict(obs, action_masks=eval_single.action_masks(), deterministic=True)
        obs, reward, done, _, _ = eval_single.step(action)
        if done and reward > 0:
            wins += 1

print(f"Trained policy win rate: {wins}/{N_EVAL} = {wins/N_EVAL:.1%}")

## 7. Watch one episode

In [ ]:
CARD_NAMES = {round(1/3, 4): "Strike", round(2/3, 4): "Defend", 1.0: "Bash", 0.0: "—"}
INTENT_NAMES = {
    0.0:              "Unknown",
    round(1/6, 4):    "Attack",
    round(2/6, 4):    "Atk+Debuff",
    round(3/6, 4):    "Atk+Block",
    round(4/6, 4):    "Buff",
    round(5/6, 4):    "Debuff",
    1.0:              "Defend",
}

env = SpireEnv()
obs, _ = env.reset(seed=999)
done = False
turn = 1

while not done:
    hp      = obs[0];  block  = obs[1] * 100;  energy = obs[2] * 3
    e_alive = obs[4 + 40]
    e_hp    = obs[4 + 40 + 1];  e_block = obs[4 + 40 + 2] * 100
    intent  = round(obs[4 + 40 + 3], 4);  idmg = obs[4 + 40 + 4] * 20

    hand = []
    for i in range(10):
        cid = round(obs[4 + i*4], 4)
        if cid > 0:
            hand.append(CARD_NAMES.get(cid, f"card{cid}"))

    print(f"Turn {turn:2d} | Player HP {hp*80:.0f}  Block {block:.0f}  Energy {energy:.0f}")
    print(f"        | Hand: {hand}")
    if e_alive > 0:
        print(f"        | Enemy HP {e_hp*100:.0f}  Block {e_block:.0f}  Intent: {INTENT_NAMES.get(intent,'?')} dmg={idmg:.0f}")

    mask = env.action_masks()
    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
    obs, reward, done, _, _ = env.step(action)
    turn += 1

outcome = "WIN" if reward > 0 else "LOSS"
print(f"\n{outcome}  (reward {reward:.2f})")